# Avito: кандидатогенерация для поиска услуг (Recall@50)

Задача: по тексту запроса и его признакам выбрать из корпуса `benchmark_items.parquet` до 50 объявлений так, чтобы в них попало как можно больше подходящих. Порядок внутри 50 не важен, считается только полнота.

Что в итоге получилось (подробности ниже по ноутбуку):

- ищем по всему корпусу, каждому объявлению считаем общий скор как взвешенную сумму нескольких сигналов;
- сигналы: BM25 по тексту объявления, BM25 по заголовку, априорная вероятность локации объявления при данной локации поиска (посчитана по train), расстояние, совпадение фильтров из `search_infm_params_text` с параметрами объявления, вероятность подкатегории по похожим запросам из train, расширение запроса заголовками объявлений, которые выбирали по похожим запросам, и косинус эмбеддингов;
- веса подобраны по Recall@50 на своей валидации (часть train) и проверены на второй отложенной части: Recall@50 = 0.907 (для сравнения, один BM25 дает около 0.42);
- в конце `answer.csv` проходит проверки из чек-листа.

Запуск: положить три parquet-файла в папку `dataset/` и выполнить ноутбук сверху вниз. На 8-ядерном CPU без GPU полный прогон с нуля занимает около 40-50 минут, из них примерно 11 минут уходит на эмбеддинги. Эмбеддинги кэшируются в `cache/`, поэтому повторный прогон быстрее.

## 0. Импорты и настройки

In [ ]:
import os
import re
import time
import hashlib

import numpy as np
import pandas as pd
import scipy.sparse as sp
import snowballstemmer
import torch
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

SEED = 42
DATA_DIR = 'dataset'
CACHE_DIR = 'cache'
EMB_MODEL_NAME = 'sergeyzh/rubert-tiny-turbo'

os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)
start_time = time.time()

## 1. Загрузка данных и проверка типов

Сначала проверяю, что файлы на месте и колонки называются так, как в описании. Отдельно смотрю на ID: они должны остаться строками длиной 16 символов. `item_id` состоит из `0-9a-f`, а у `query_id` в данных встречаются и заглавные латинские буквы, поэтому для него проверяю только длину и тип.

In [ ]:
for name in ['train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet']:
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Не найден файл {path}')

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
queries = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_queries.parquet'))
items = pd.read_parquet(os.path.join(DATA_DIR, 'benchmark_items.parquet'))
print('train:', train.shape)
print('queries:', queries.shape)
print('items:', items.shape)

train: (497673, 19)
queries: (2452, 6)
items: (189212, 14)


In [ ]:
search_cols = ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
item_cols = ['item_id', 'item_title_raw', 'item_description_raw', 'item_infm_params_text', 'item_category_id',
             'item_microcat_id', 'item_price', 'item_rating', 'item_rating_reviews_count', 'item_location_id',
             'item_latitude', 'item_longitude', 'item_is_phone_hidden', 'item_is_message_forbidden']

missing = {
    'train': set(search_cols + item_cols) - set(train.columns),
    'queries': set(['query_id'] + search_cols) - set(queries.columns),
    'items': set(item_cols) - set(items.columns),
}
print(missing)
assert all(len(m) == 0 for m in missing.values())

{'train': set(), 'queries': set(), 'items': set()}


In [ ]:
def check_ids(ids, name, hex_only):
    assert ids.map(type).eq(str).all(), name
    assert ids.str.len().eq(16).all(), name
    if hex_only:
        assert ids.str.fullmatch('[0-9a-f]{16}').all(), name
    print(f'{name}: dtype={ids.dtype}, пример={ids.iloc[0]}, пропусков={ids.isna().sum()}')


check_ids(queries['query_id'], 'query_id', hex_only=False)
check_ids(items['item_id'], 'item_id (корпус)', hex_only=True)
check_ids(train['item_id'], 'item_id (train)', hex_only=True)

query_id: dtype=str, пример=70DfDUpwjxB4lzFd, пропусков=0
item_id (корпус): dtype=str, пример=111eb8b979577d79, пропусков=0
item_id (train): dtype=str, пример=e8b685dffe1a408e, пропусков=0


Текстовые поля иногда пустые (например, 35 описаний в корпусе). Чтобы дальше ничего не падало на `None`, сразу заменяю пропуски в текстах на пустую строку.

In [ ]:
for df in [train, queries]:
    for col in ['search_query', 'search_infm_params_text']:
        df[col] = df[col].fillna('')
for df in [train, items]:
    for col in ['item_title_raw', 'item_description_raw', 'item_infm_params_text']:
        df[col] = df[col].fillna('')

print(queries.isna().sum().sum(), items[['item_title_raw', 'item_description_raw', 'item_infm_params_text']].isna().sum().sum())

0 0


## 2. EDA

In [ ]:
print('строк в train:', len(train))
print('уникальных текстов запросов в train:', train['search_query'].nunique())
print('уникальных комбинаций всех search_* полей в train:', train[search_cols].drop_duplicates().shape[0])
print('уникальных объявлений в train:', train['item_id'].nunique())
print()
print('запросов в benchmark:', len(queries), '| уникальных query_id:', queries['query_id'].nunique(), '| уникальных текстов:', queries['search_query'].nunique())
print('объявлений в корпусе:', len(items), '| уникальных item_id:', items['item_id'].nunique())
print('объявлений корпуса, которые встречаются в train:', items['item_id'].isin(set(train['item_id'])).sum())
seen_share = queries['search_query'].isin(set(train['search_query'])).mean()
print('доля запросов benchmark, текст которых был в train:', round(seen_share, 3))

строк в train: 497673
уникальных текстов запросов в train: 74529
уникальных комбинаций всех search_* полей в train: 354463


уникальных объявлений в train: 344825

запросов в benchmark: 2452 | уникальных query_id: 2452 | уникальных текстов: 2452
объявлений в корпусе: 189212 | уникальных item_id: 189212


объявлений корпуса, которые встречаются в train: 18142


доля запросов benchmark, текст которых был в train: 0.37


In [ ]:
lengths = pd.DataFrame({
    'слов в запросе': queries['search_query'].str.split().str.len().describe(),
    'символов в заголовке': items['item_title_raw'].str.len().describe(),
    'символов в описании': items['item_description_raw'].str.len().describe(),
    'символов в параметрах': items['item_infm_params_text'].str.len().describe(),
})
lengths.round(1)

,слов в запросе,символов в заголовке,символов в описании,символов в параметрах
count,2452.0,189212.0,189212.0,189212.0
mean,3.2,33.6,1404.9,973.9
std,1.3,12.2,1284.3,875.9
min,1.0,1.0,0.0,10.0
25%,2.0,24.0,422.0,453.0
50%,3.0,35.0,1054.0,738.0
75%,4.0,44.0,1978.0,1206.0
max,10.0,100.0,8494.0,19068.0


In [ ]:
print('пустой search_infm_params_text в train:', round((train['search_infm_params_text'] == '').mean(), 3))
print('пустой search_infm_params_text в benchmark:', round((queries['search_infm_params_text'] == '').mean(), 3))
print('пустых описаний в корпусе:', (items['item_description_raw'] == '').sum())
print('пустых заголовков в корпусе:', (items['item_title_raw'] == '').sum())
print('запросов из одного слова в benchmark:', (queries['search_query'].str.split().str.len() == 1).sum())

пустой search_infm_params_text в train: 0.33
пустой search_infm_params_text в benchmark: 0.631
пустых описаний в корпусе: 35
пустых заголовков в корпусе: 0
запросов из одного слова в benchmark: 109


In [ ]:
print(pd.crosstab(train['search_category'], train['item_category_id']))
print()
print('search_category в benchmark:', queries['search_category'].value_counts().to_dict())
print('item_category_id в корпусе (топ-10):', items['item_category_id'].value_counts().head(10).to_dict())

item_category_id  10   19   27   28   40   81   112     114
search_category                                            
0                   0    0    0    0    0    0    0      34
10                  0    0    0    0    0    0    0       2
81                  0    0    0    0    0    0    0       2
114                 1    2    4    2    2    2    2  497620

search_category в benchmark: {114: 2230, 0: 222}
item_category_id в корпусе (топ-10): {114: 187336, 19: 357, 112: 141, 40: 122, 33: 103, 10: 98, 20: 84, 25: 82, 36: 74, 111: 74}


In [ ]:
print('дублей строк (запрос + объявление) в train:', train.duplicated(search_cols + ['item_id']).sum())
print('объявлений в корпусе с таким же заголовком и описанием, как у другого:', items.duplicated(['item_title_raw', 'item_description_raw']).sum())

дублей строк (запрос + объявление) в train: 30619


объявлений в корпусе с таким же заголовком и описанием, как у другого: 18754


In [ ]:
same_loc = (train['search_location_id'] == train['item_location_id']).mean()
print('доля пар в train, где локация поиска совпадает с локацией объявления:', round(same_loc, 3))
print('локации поиска, у которых чаще всего выбирают объявление из другой локации:')
print(train.loc[train['search_location_id'] != train['item_location_id'], 'search_location_id'].value_counts().head(5))

доля пар в train, где локация поиска совпадает с локацией объявления: 0.831
локации поиска, у которых чаще всего выбирают объявление из другой локации:
search_location_id
107620    12277
621540     6839
107621     6310
637640     2873
637680     1528
Name: count, dtype: int64


In [ ]:
train['search_infm_params_text'].value_counts().head(12)

search_infm_params_text
                                                                                                        163999
Вид услуги Автосервис, аренда Тип услуги Автосервис                                                      19654
Вид услуги Ремонт и отделка                                                                              15586
Вид услуги Красота, здоровье                                                                             15355
Вид услуги Праздники, мероприятия                                                                        12474
Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Тип услуги Автосервис     11644
Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье                                                 11589
Вид услуги Обучение, курсы                                                                               11080
Вид услуги                                                                              

Что видно из EDA:

- Почти все запросы и объявления в train из категории 114 (услуги). В корпусе есть около 1.9 тыс. объявлений из других категорий, это скорее шум. У 222 запросов benchmark `search_category = 0`, но в train у таких запросов объявления все равно из 114.
- Запросы короткие, медиана 3 слова. Описания объявлений длинные, в параметрах много служебного текста (график работы и т.п.).
- Все тексты запросов в benchmark разные, и только около 37% из них встречались в train. Значит, простое запоминание пар "запрос - объявление" не спасет, нужен поиск по тексту.
- Объявления корпуса почти не пересекаются с train (около 18 тыс. из 189 тыс.).
- Очень сильный сигнал - локация: в 83% пар локация поиска совпадает с локацией объявления. Остальное в основном "регион -> город" (например, поиск по области, а объявление в областном центре). Такие переходы можно посчитать по train.
- `search_infm_params_text` - это фильтры поиска ("Вид услуги ...", "Тип услуги ..."). У 37% запросов benchmark они заполнены, и их можно сверять с `item_infm_params_text`.
- Есть дубли строк в train и объявления с одинаковым текстом в корпусе. Дубли строк при группировке по запросу схлопываются, а дубли объявлений проверю в конце на ответе.

## 3. Подготовка текста

Токенизация простая: нижний регистр, `ё -> е`, слова из букв и цифр, затем стемминг Snowball для русского (чтобы "дровах" и "дрова" совпали). Эмодзи и прочие символы просто выкидываются, пустая строка дает пустой результат.

In [ ]:
stemmer = snowballstemmer.stemmer('russian')
token_re = re.compile(r'[a-zа-я0-9]+')
stem_cache = {}


def stem_text(text):
    words = token_re.findall(text.lower().replace('ё', 'е'))
    result = []
    for w in words:
        if w not in stem_cache:
            stem_cache[w] = stemmer.stemWord(w)
        result.append(stem_cache[w])
    return ' '.join(result)


examples = ['Баня на дровах', 'ремонт iPhone 11 \U0001F600', '', '   ', '№1!!! Ёлка-палка, 5-10 м²']
for example in examples:
    print(repr(stem_text(example)))

'бан на дров'
'ремонт iphone 11'
''
''
'1 елк палк 5 10 м'


## 4. Валидация: сплит и метрика

У benchmark нет разметки, поэтому качество проверяю на части train.

- Делю по уникальным комбинациям всех `search_*` полей, а не по строкам: одна и та же комбинация не может оказаться и в обучающей, и в валидационной части. 12% комбинаций уходит в валидацию.
- Все объявления, выбранные по одной комбинации, собираю в один валидационный запрос с несколькими релевантными `item_id`.
- В benchmark у каждого запроса свой текст, и только 37% текстов были в train. Чтобы валидация была похожа, беру по одной комбинации на текст и набираю запросы так, чтобы доля "виденных" текстов (есть в обучающей части) тоже была около 37%.
- Делаю две непересекающиеся выборки по ~3000 запросов: A для подбора весов и B для честной проверки.
- Корпус для поиска на валидации: весь `benchmark_items` плюс релевантные объявления валидационных запросов. То есть нужные объявления ищутся среди ~190 тыс. других, как и в настоящей задаче.
- Вся статистика (переходы локаций, похожие запросы и т.д.) на валидации считается только по обучающей части.

In [ ]:
def recall_at_50(predictions, relevant):
    scores = []
    for pred, rel in zip(predictions, relevant):
        rel = set(rel)
        scores.append(len(set(pred[:50]) & rel) / len(rel))
    return float(np.mean(scores))


example_pred = [['a1'], ['b1', 'x1'], ['z1']]
example_rel = [['a1'], ['b1', 'b2'], ['c1']]
print('пример из задания:', recall_at_50(example_pred, example_rel))
assert abs(recall_at_50(example_pred, example_rel) - 0.5) < 1e-12
assert recall_at_50([[str(i) for i in range(100)]], [['0', '75']]) == 0.5
assert recall_at_50([['a', 'a', 'b']], [['a', 'b']]) == 1.0

пример из задания: 0.5


In [ ]:
combos = train[search_cols].drop_duplicates().reset_index(drop=True)
rng = np.random.RandomState(SEED)
combos['is_val'] = rng.rand(len(combos)) < 0.12
train = train.merge(combos, on=search_cols, how='left')
assert train['is_val'].notna().all() and train['item_id'].dtype == items['item_id'].dtype

train_part = train[~train['is_val']].reset_index(drop=True)
val_part = train[train['is_val']].reset_index(drop=True)
print('обучающая часть:', len(train_part), 'строк | валидационная часть:', len(val_part), 'строк')

val_queries = val_part.groupby(search_cols)['item_id'].apply(lambda s: sorted(set(s))).reset_index(name='relevant')
val_queries = val_queries.sample(frac=1, random_state=SEED).drop_duplicates('search_query').reset_index(drop=True)
val_queries['seen'] = val_queries['search_query'].isin(set(train_part['search_query']))

n_unseen = 1900
n_seen = int(round(n_unseen * seen_share / (1 - seen_share)))
unseen_q = val_queries[~val_queries['seen']]
seen_q = val_queries[val_queries['seen']]
val_a = pd.concat([unseen_q.iloc[:n_unseen], seen_q.iloc[:n_seen]]).reset_index(drop=True)
val_b = pd.concat([unseen_q.iloc[n_unseen:2 * n_unseen], seen_q.iloc[n_seen:2 * n_seen]]).reset_index(drop=True)
print('A:', len(val_a), 'запросов, доля виденных', round(val_a['seen'].mean(), 3), ', релевантных на запрос', round(val_a['relevant'].str.len().mean(), 2))
print('B:', len(val_b), 'запросов, доля виденных', round(val_b['seen'].mean(), 3), ', релевантных на запрос', round(val_b['relevant'].str.len().mean(), 2))
assert not set(val_a['search_query']) & set(val_b['search_query'])

обучающая часть: 439149 строк | валидационная часть: 58524 строк


A: 3015 запросов, доля виденных 0.37 , релевантных на запрос 1.12
B: 3015 запросов, доля виденных 0.37 , релевантных на запрос 1.1


In [ ]:
val_relevant_ids = set(i for lst in pd.concat([val_a, val_b])['relevant'] for i in lst)
extra_items = val_part[val_part['item_id'].isin(val_relevant_ids)].drop_duplicates('item_id')[item_cols]
extra_items = extra_items[~extra_items['item_id'].isin(set(items['item_id']))]
all_corpus = pd.concat([items[item_cols], extra_items]).reset_index(drop=True)
n_bench_items = len(items)
assert all_corpus['item_id'].is_unique
assert (all_corpus['item_id'].iloc[:n_bench_items].values == items['item_id'].values).all()
print('корпус для валидации:', len(all_corpus), '| из них добавлено из train:', len(extra_items))

корпус для валидации: 195408 | из них добавлено из train: 6196


Стемминг текстов корпуса. Первые 189 212 строк `all_corpus` - это ровно `benchmark_items`, поэтому для финального ответа потом просто беру эту часть.

In [ ]:
t = time.time()
all_corpus['title_st'] = [stem_text(x) for x in all_corpus['item_title_raw']]
all_corpus['full_st'] = [a + ' ' + stem_text(b) + ' ' + stem_text(c) for a, b, c in
                         zip(all_corpus['title_st'], all_corpus['item_description_raw'], all_corpus['item_infm_params_text'])]
title_st_map = {}
for item_id, title in zip(train['item_id'], train['item_title_raw']):
    if item_id not in title_st_map:
        title_st_map[item_id] = stem_text(title)
train['title_st'] = train['item_id'].map(title_st_map)
train_part['title_st'] = train_part['item_id'].map(title_st_map)
for df in [val_a, val_b, queries]:
    df['query_st'] = [stem_text(x) for x in df['search_query']]
print('стемминг занял', round(time.time() - t), 'сек')

стемминг занял 63 сек


## 5. Эмбеддинги

Беру открытую модель `sergeyzh/rubert-tiny-turbo` (маленький русский BERT, 312-мерные эмбеддинги). Она быстрая на CPU, а большие модели вроде multilingual-e5 на этом железе считали бы корпус больше часа. Модель скачивается с Hugging Face один раз при первом запуске, дальше все считается локально без обращений к сети.

Текст объявления для эмбеддинга: заголовок + первые 300 символов описания. Результаты сохраняю в `cache/`, в имя файла добавляю хэш от входных текстов, чтобы случайно не взять устаревший кэш.

In [ ]:
emb_model = SentenceTransformer(EMB_MODEL_NAME, device='cpu')
emb_model.max_seq_length = 128


def encode(texts, name):
    key = hashlib.md5('\n'.join(texts).encode('utf-8')).hexdigest()[:12]
    path = os.path.join(CACHE_DIR, f'{name}_{key}.npy')
    if os.path.exists(path):
        return np.load(path)
    emb = emb_model.encode(texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    emb = emb.astype(np.float32)
    np.save(path, emb)
    return emb


t = time.time()
item_texts = (all_corpus['item_title_raw'] + '. ' + all_corpus['item_description_raw'].str.slice(0, 300)).tolist()
item_emb = encode(item_texts, 'items')
all_query_texts = sorted(set(train['search_query']) | set(val_a['search_query']) | set(val_b['search_query']) | set(queries['search_query']))
query_emb = encode(all_query_texts, 'queries')
query_emb_pos = pd.Series(np.arange(len(all_query_texts)), index=all_query_texts)
print(item_emb.shape, query_emb.shape, 'за', round(time.time() - t), 'сек')

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

(195408, 312) (76074, 312) за 2 сек


## 6. Сигналы для скоринга

Для каждого запроса считаю скор для всех объявлений корпуса сразу (матрицами, батчами по 200 запросов) и беру 50 лучших. Все, что считается по train, собирается функцией `build_index`: на валидации ей передается только обучающая часть, а для финального ответа - весь train.

Сигналы:

1. `bm_full` - BM25 по заголовку + описанию + параметрам (стеммированные слова).
2. `bm_title` - BM25 только по заголовку, заголовок самый "плотный" по смыслу.
3. `loc` - вероятность локации объявления при локации поиска, посчитанная по train: P(item_location | search_location). В скор идет её логарифм. Для локации поиска, которой нет в train, строка нулевая, и этот сигнал просто одинаковый для всех объявлений.
4. `dist` - расстояние (км) от "центра" локации поиска (медиана координат выбранных объявлений в train) до координат объявления. Помогает, когда пара локаций в train не встречалась, но объявление рядом.
5. `flt` - фильтры: из `search_infm_params_text` достаю "Вид услуги ..." и "Тип услуги ..." и проверяю, есть ли они в `item_infm_params_text`. +1 если все совпали, -1 если нет, 0 если фильтров нет.
6. `mc` - подкатегория: нахожу 20 самых похожих запросов из train (по эмбеддингам) и смотрю, из каких `item_microcat_id` там выбирали объявления. Получается распределение подкатегорий для запроса.
7. `exp` - расширение запроса: по тем же похожим запросам склеиваю заголовки выбранных объявлений и ищу похожие заголовки (TF-IDF косинус). Так находятся синонимы, например "ассенизаторские услуги" -> "откачка септика".
8. `emb` - косинус эмбеддингов запроса и объявления.
9. `cat` - объявление из категории 114 (услуги) или нет.

Итоговый скор - взвешенная сумма этих сигналов, веса подбираются ниже по Recall@50.

In [ ]:
def bm25_matrix(texts, k1=1.2, b=0.75):
    vectorizer = CountVectorizer(analyzer=str.split, dtype=np.float32)
    tf = vectorizer.fit_transform(texts).tocsr()
    n_docs = tf.shape[0]
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    doc_len = np.asarray(tf.sum(axis=1)).ravel()
    avg_len = max(doc_len.mean(), 1.0)
    rows = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    x = tf.data
    tf.data = (x * (k1 + 1) / (x + k1 * (1 - b + b * doc_len[rows] / avg_len)) * idf[tf.indices]).astype(np.float32)
    return tf.T.tocsr(), vectorizer.vocabulary_


def query_matrix(texts, vocabulary):
    return CountVectorizer(analyzer=str.split, vocabulary=vocabulary, binary=True, dtype=np.float32).transform(texts)


def to_float(values):
    return pd.Series(values).astype(float).values


def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

In [ ]:
all_params = pd.concat([train['item_infm_params_text'], items['item_infm_params_text']])
stop_words = r'(?: Место оказания| Тип услуги| Вид услуги| Тип стоимости|$)'
filter_values = {}
for key in ['Вид услуги', 'Тип услуги']:
    counts = all_params.str.extract(key + r' (.*?)' + stop_words)[0].value_counts()
    filter_values[key] = sorted([v for v in counts[counts >= 5].index if v and 'Место' not in v], key=len, reverse=True)


def parse_filters(params_text):
    found = []
    for key, values in filter_values.items():
        for v in values:
            if key + ' ' + v in params_text:
                found.append(key + ' ' + v)
                break
    return found


print({k: len(v) for k, v in filter_values.items()})
for text in ['', 'Вид услуги', 'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье', 'Рейтинг пользователя 4 звезды и выше']:
    print(repr(text), '->', parse_filters(text))

{'Вид услуги': 26, 'Тип услуги': 2461}
'' -> []
'Вид услуги' -> []
'Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье' -> ['Вид услуги Красота, здоровье', 'Тип услуги Маникюр, педикюр']
'Рейтинг пользователя 4 звезды и выше' -> []


In [ ]:
sample = train.sample(50000, random_state=SEED)
sample_filters = sample['search_infm_params_text'].map({t: parse_filters(t) for t in sample['search_infm_params_text'].unique()})
has_filters = sample_filters.str.len() > 0
match = [all(f in p for f in fs) for fs, p in zip(sample_filters[has_filters], sample.loc[has_filters, 'item_infm_params_text'])]
print('строк с фильтрами:', round(has_filters.mean(), 3), '| фильтры совпали с параметрами объявления:', round(np.mean(match), 3))

строк с фильтрами: 0.644 | фильтры совпали с параметрами объявления: 0.979


In [ ]:
def build_index(train_df, corpus, corpus_emb, k_neighbors=20):
    index = {'corpus': corpus, 'item_ids': corpus['item_id'].to_numpy(dtype=object), 'emb': corpus_emb, 'k': k_neighbors}
    index['bm_full'], index['vocab_full'] = bm25_matrix(corpus['full_st'].values)
    index['bm_title'], index['vocab_title'] = bm25_matrix(corpus['title_st'].values, b=0.5)

    search_locs = pd.Index(sorted(train_df['search_location_id'].unique()))
    item_locs = pd.Index(sorted(set(train_df['item_location_id']) | set(corpus['item_location_id'])))
    counts = train_df.groupby(['search_location_id', 'item_location_id']).size().reset_index(name='n')
    loc_prior = np.zeros((len(search_locs) + 1, len(item_locs)), dtype=np.float32)
    loc_prior[search_locs.get_indexer(counts['search_location_id']), item_locs.get_indexer(counts['item_location_id'])] = counts['n'].values
    loc_prior = loc_prior / np.maximum(loc_prior.sum(axis=1, keepdims=True), 1)
    index['search_locs'] = search_locs
    index['loc_prior'] = loc_prior
    index['item_loc_pos'] = item_locs.get_indexer(corpus['item_location_id'])

    lat = to_float(corpus['item_latitude'])
    lon = to_float(corpus['item_longitude'])
    loc_center = pd.DataFrame({'loc': corpus['item_location_id'].values, 'lat': lat, 'lon': lon}).groupby('loc')[['lat', 'lon']].median()
    index['item_lat'] = np.where(np.isnan(lat), loc_center['lat'].reindex(corpus['item_location_id']).values, lat)
    index['item_lon'] = np.where(np.isnan(lon), loc_center['lon'].reindex(corpus['item_location_id']).values, lon)
    clicks = pd.DataFrame({'loc': train_df['search_location_id'].values,
                           'lat': to_float(train_df['item_latitude']), 'lon': to_float(train_df['item_longitude'])})
    index['search_center'] = clicks.groupby('loc')[['lat', 'lon']].median()

    past = train_df.groupby('search_query').agg(titles=('title_st', ' '.join), microcats=('item_microcat_id', list)).reset_index()
    index['past_emb'] = query_emb[query_emb_pos.loc[past['search_query']].values]
    title_tfidf = TfidfVectorizer(analyzer=str.split, sublinear_tf=True, min_df=2, dtype=np.float32)
    title_tfidf.fit(corpus['title_st'])
    index['title_vecs'] = normalize(title_tfidf.transform(corpus['title_st'])).T.tocsr()
    index['past_titles'] = normalize(title_tfidf.transform(past['titles']))

    microcats = pd.Index(sorted(train_df['item_microcat_id'].unique()))
    rows = np.repeat(np.arange(len(past)), past['microcats'].str.len())
    cols = microcats.get_indexer(np.concatenate(past['microcats'].values))
    mc = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(past), len(microcats)))
    index['past_microcats'] = normalize(mc, norm='l1')
    item_mc_pos = microcats.get_indexer(corpus['item_microcat_id'])
    item_mc_pos[item_mc_pos < 0] = len(microcats)
    index['item_mc_pos'] = item_mc_pos

    index['is_services'] = (corpus['item_category_id'].values == 114).astype(np.float32)
    index['filter_cache'] = {}
    return index

In [ ]:
def filter_scores(index, params_texts):
    result = np.zeros((len(params_texts), len(index['item_ids'])), dtype=np.float32)
    for row, text in enumerate(params_texts):
        filters = parse_filters(text)
        if not filters:
            continue
        ok = np.ones(len(index['item_ids']), dtype=bool)
        for f in filters:
            if f not in index['filter_cache']:
                index['filter_cache'][f] = index['corpus']['item_infm_params_text'].str.contains(f, regex=False).values
            ok &= index['filter_cache'][f]
        result[row] = np.where(ok, 1, -1)
    return result


def compute_signals(index, batch):
    signals = {}
    signals['bm_full'] = (query_matrix(batch['query_st'], index['vocab_full']) @ index['bm_full']).toarray()
    signals['bm_title'] = (query_matrix(batch['query_st'], index['vocab_title']) @ index['bm_title']).toarray()

    q_emb = query_emb[query_emb_pos.loc[batch['search_query']].values]
    signals['emb'] = q_emb @ index['emb'].T

    k = index['k']
    sims = q_emb @ index['past_emb'].T
    nearest = np.argpartition(-sims, k, axis=1)[:, :k]
    weights = np.clip(np.take_along_axis(sims, nearest, axis=1), 0, None) ** 2
    rows = np.repeat(np.arange(len(batch)), k)
    neighbors = sp.csr_matrix((weights.ravel(), (rows, nearest.ravel())), shape=sims.shape)
    signals['exp'] = (normalize(neighbors @ index['past_titles']) @ index['title_vecs']).toarray()
    mc = normalize(neighbors @ index['past_microcats'], norm='l1').toarray()
    mc = np.hstack([mc, np.zeros((len(batch), 1), dtype=np.float32)])
    signals['mc'] = mc[:, index['item_mc_pos']]

    loc_pos = index['search_locs'].get_indexer(batch['search_location_id'])
    loc_pos[loc_pos < 0] = len(index['search_locs'])
    signals['loc'] = index['loc_prior'][loc_pos][:, index['item_loc_pos']]
    center = index['search_center'].reindex(batch['search_location_id'])
    dist = haversine(center['lat'].values[:, None], center['lon'].values[:, None], index['item_lat'][None, :], index['item_lon'][None, :])
    signals['dist'] = np.nan_to_num(dist, nan=0.0).astype(np.float32)

    signals['flt'] = filter_scores(index, batch['search_infm_params_text'].values)
    signals['cat'] = np.broadcast_to(index['is_services'], signals['emb'].shape)
    return signals


def combine(s, w):
    score = w['bm_full'] * s['bm_full'] + w['bm_title'] * s['bm_title'] + w['exp'] * s['exp'] + w['emb'] * s['emb']
    score = score + w['loc'] * np.log(s['loc'] + w['loc_eps']) - w['dist'] * np.log1p(s['dist'] / w['dist_scale'])
    score = score + w['mc'] * np.log(s['mc'] + w['mc_eps']) + w['flt'] * s['flt'] + w['cat'] * s['cat']
    return score


def top_k(score, k=50):
    k = min(k, score.shape[1])
    part = np.argpartition(-score, k - 1, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(score, part, axis=1), axis=1, kind='stable')
    return np.take_along_axis(part, order, axis=1)

Функция прогона: для батча считает сигналы один раз, а дальше для каждого набора весов берет топ-50. Если передать `pool_weights`, она еще сохраняет "пул кандидатов" для подбора весов: объединение топ-300 по нескольким разным наборам весов вместе со значениями всех сигналов. Потом веса перебираются уже только внутри пула, это в сотни раз быстрее, чем пересчитывать весь корпус.

In [ ]:
SIGNALS = ['bm_full', 'bm_title', 'exp', 'emb', 'mc', 'loc', 'dist', 'flt', 'cat']


def run(index, qdf, weight_sets, pool_weights=None, pool_size=300, batch_size=200):
    predictions = {name: [] for name in weight_sets}
    pool = []
    for start in range(0, len(qdf), batch_size):
        batch = qdf.iloc[start:start + batch_size]
        s = compute_signals(index, batch)
        for name, w in weight_sets.items():
            top = top_k(combine(s, w))
            predictions[name].extend([list(index['item_ids'][row]) for row in top])
        if pool_weights is not None:
            cand = np.concatenate([top_k(combine(s, w), pool_size) for w in pool_weights], axis=1)
            for row in range(len(batch)):
                c = np.unique(cand[row])
                part = pd.DataFrame({name: s[name][row, c] for name in SIGNALS})
                part['q'] = start + row
                part['item_id'] = index['item_ids'][c]
                pool.append(part)
    if pool_weights is not None:
        return predictions, pd.concat(pool, ignore_index=True)
    return predictions

## 7. Пошаговая проверка на валидации

Собираю индекс по обучающей части и прогоняю выборки A и B. Начальные веса выставлены вручную по порядку величин сигналов (BM25 порядка 5-20, косинусы 0-1, логарифм вероятности локации от -10 до 0). Каждая следующая строка добавляет один сигнал, чтобы было видно, помогает ли он.

In [ ]:
t = time.time()
val_index = build_index(train_part, all_corpus, item_emb)
print('индекс для валидации собран за', round(time.time() - t), 'сек')

zero = dict(bm_full=0, bm_title=0, exp=0, emb=0, loc=0, loc_eps=1e-4, dist=0, dist_scale=30, mc=0, mc_eps=1e-3, flt=0, cat=0)
steps = {}
steps['1. BM25 по тексту'] = dict(zero, bm_full=1)
steps['2. + BM25 по заголовку'] = dict(steps['1. BM25 по тексту'], bm_title=0.5)
steps['3. + локация'] = dict(steps['2. + BM25 по заголовку'], loc=2)
steps['4. + фильтры'] = dict(steps['3. + локация'], flt=2)
steps['5. + подкатегория'] = dict(steps['4. + фильтры'], mc=1)
steps['6. + эмбеддинги'] = dict(steps['5. + подкатегория'], emb=10)
steps['7. + расширение запроса'] = dict(steps['6. + эмбеддинги'], exp=5)
steps['8. + расстояние'] = dict(steps['7. + расширение запроса'], dist=0.5)
pool_weights = [
    steps['3. + локация'],
    dict(zero, emb=20, loc=2),
    dict(zero, exp=20, loc=2, flt=2),
    steps['8. + расстояние'],
]

val_ab = pd.concat([val_a.assign(part='A'), val_b.assign(part='B')]).reset_index(drop=True)
t = time.time()
val_pred, pool = run(val_index, val_ab, steps, pool_weights=pool_weights)
print('прогон валидации занял', round(time.time() - t), 'сек, строк в пуле:', len(pool))

is_a = (val_ab['part'] == 'A').values
rows = []
for name, pred in val_pred.items():
    rows.append({'шаг': name,
                 'Recall@50 A': recall_at_50([p for p, f in zip(pred, is_a) if f], val_ab.loc[is_a, 'relevant']),
                 'Recall@50 B': recall_at_50([p for p, f in zip(pred, is_a) if not f], val_ab.loc[~is_a, 'relevant'])})
pd.DataFrame(rows).round(4)

индекс для валидации собран за 37 сек


прогон валидации занял 888 сек, строк в пуле: 4444868


,шаг,Recall@50 A,Recall@50 B
0,1. BM25 по тексту,0.3920,0.3921
1,2. + BM25 по заголовку,0.4235,0.4354
2,3. + локация,0.8732,0.8504
3,4. + фильтры,0.8916,0.8703
4,5. + подкатегория,0.8974,0.8927
5,6. + эмбеддинги,0.9003,0.8918
6,7. + расширение запроса,0.9021,0.8926
7,8. + расстояние,0.9085,0.9003


## 8. Подбор весов по валидации

Перебор по координатам: беру один вес, пробую несколько значений, оставляю лучшее по Recall@50 на выборке A, перехожу к следующему. Три полных прохода. Выборку B при подборе не трогаю и смотрю на нее только в конце.

Метрика внутри пула считается так же, как обычная: сортирую кандидатов запроса по скору, беру 50 лучших и делю число найденных релевантных на число всех релевантных запроса (включая те, что в пул не попали).

In [ ]:
relevant_sets = [set(r) for r in val_ab['relevant']]
pool['rel'] = [item in relevant_sets[q] for q, item in zip(pool['q'], pool['item_id'])]
n_relevant = val_ab['relevant'].str.len().values
pool_q = pool['q'].values
pool_rel = pool['rel'].values
pool_signals = {name: pool[name].values for name in SIGNALS}
print('доля релевантных, которые вообще попали в пул:', round(pool['rel'].sum() / n_relevant.sum(), 4))


def pool_recall(w):
    score = combine(pool_signals, w)
    order = np.lexsort((-score, pool_q))
    q_sorted = pool_q[order]
    starts = np.r_[0, np.flatnonzero(np.diff(q_sorted)) + 1]
    rank = np.arange(len(order)) - np.repeat(starts, np.diff(np.r_[starts, len(order)]))
    hits = np.bincount(q_sorted[(rank < 50) & pool_rel[order]], minlength=len(val_ab))
    recall = hits / n_relevant
    return recall[is_a].mean(), recall[~is_a].mean()


grid = {
    'bm_full': [0.5, 1, 1.5, 2],
    'bm_title': [0, 0.25, 0.5, 1],
    'exp': [0, 2.5, 5, 10, 15],
    'emb': [0, 10, 20, 30, 40, 60],
    'loc': [1, 1.5, 2, 2.5, 3],
    'loc_eps': [1e-5, 3e-5, 1e-4, 3e-4, 1e-3],
    'dist': [0, 0.5, 1, 2, 3, 5],
    'dist_scale': [10, 30, 100, 300, 1000],
    'mc': [0, 0.25, 0.5, 1, 2],
    'mc_eps': [1e-4, 1e-3, 1e-2],
    'flt': [0, 1, 2, 3, 5, 8],
    'cat': [0, 1, 2],
}

best_w = dict(steps['8. + расстояние'])
best_score = pool_recall(best_w)[0]
print('старт: A =', round(best_score, 4))
for iteration in range(3):
    for name, values in grid.items():
        for v in values:
            w = dict(best_w, **{name: v})
            score = pool_recall(w)[0]
            if score > best_score + 1e-6:
                best_score, best_w = score, w
    print(f'проход {iteration + 1}: A = {best_score:.4f}')
print(best_w)

доля релевантных, которые вообще попали в пул: 0.9784


старт: A = 0.9085


проход 1: A = 0.9205


проход 2: A = 0.9208


проход 3: A = 0.9208
{'bm_full': 1, 'bm_title': 0, 'exp': 5, 'emb': 40, 'loc': 2, 'loc_eps': 3e-05, 'dist': 1, 'dist_scale': 30, 'mc': 0.5, 'mc_eps': 0.001, 'flt': 5, 'cat': 0}


In [ ]:
a, b = pool_recall(best_w)
rows = [{'вариант': 'все сигналы', 'A': a, 'B': b}]
for name in ['bm_full', 'bm_title', 'exp', 'emb', 'mc', 'loc', 'dist', 'flt', 'cat']:
    a, b = pool_recall(dict(best_w, **{name: 0}))
    rows.append({'вариант': 'без ' + name, 'A': a, 'B': b})
pd.DataFrame(rows).round(4)

,вариант,A,B
0,все сигналы,0.9208,0.9066
1,без bm_full,0.8436,0.8237
2,без bm_title,0.9208,0.9066
3,без exp,0.9180,0.9034
4,без emb,0.9091,0.8978
5,без mc,0.9198,0.9001
6,без loc,0.7134,0.7108
7,без dist,0.9144,0.9037
8,без flt,0.9118,0.8922
9,без cat,0.9208,0.9066


Проверка на отложенной выборке B уже полным прогоном по всему корпусу (без пула), с подобранными весами. Заодно смотрю распределение числа кандидатов, качество на виденных и новых текстах и по длине запроса.

In [ ]:
t = time.time()
pred_b = run(val_index, val_b, {'best': best_w})['best']
val_b['pred'] = pd.Series(pred_b, index=val_b.index)
val_b['recall'] = [len(set(p) & set(r)) / len(r) for p, r in zip(pred_b, val_b['relevant'])]
print('Recall@50 на B (полный прогон):', round(recall_at_50(pred_b, val_b['relevant']), 4), '| время', round(time.time() - t), 'сек')
print('виденные тексты:', round(val_b.loc[val_b['seen'], 'recall'].mean(), 4), '| новые тексты:', round(val_b.loc[~val_b['seen'], 'recall'].mean(), 4))
print('кандидатов на запрос:', pd.Series([len(p) for p in pred_b]).value_counts().to_dict())
print('запросов с дублями внутри ответа:', sum(len(p) != len(set(p)) for p in pred_b))
val_b['n_words'] = val_b['search_query'].str.split().str.len().clip(upper=5)
val_b.groupby('n_words')['recall'].agg(['mean', 'size']).round(3)

Recall@50 на B (полный прогон): 0.9066 | время 82 сек
виденные тексты: 0.9247 | новые тексты: 0.896
кандидатов на запрос: {50: 3015}
запросов с дублями внутри ответа: 0


,mean,size
n_words,,
1,0.837,183
2,0.910,969
3,0.922,950
4,0.904,580
5,0.896,333


Несколько примеров: запрос, релевантное объявление и первые 5 кандидатов из нашего топ-50.

In [ ]:
titles = pd.Series(all_corpus['item_title_raw'].values, index=all_corpus['item_id'].values)
for _, row in val_b.sample(3, random_state=1).iterrows():
    print('ЗАПРОС:', row['search_query'], '| фильтры:', row['search_infm_params_text'] or '-', '| recall:', row['recall'])
    print('   нужно:', [titles[i] for i in row['relevant']])
    print('   топ-5:', [titles[i] for i in row['pred'][:5]])
    print()

ЗАПРОС: печать принта на одежде | фильтры: - | recall: 1.0
   нужно: ['Печать принта на одежде']
   топ-5: ['Печать на футболках и худи любой сложности', 'Флаги на заказ / Печать флагов / Сублимация', 'Печать принта на одежде', 'Печать на футболке и одежде', 'Печать логотипов/ Дтф печать на футболке']

ЗАПРОС: подсобный рабочий женщина | фильтры: - | recall: 0.0
   нужно: ['Бригада помощниц по дому']
   топ-5: ['Разнорабочие - подсобники', 'Промышленные бетонные полы, стяжка', 'Разнорабочие Подсобник Разнорабочий Услуги', 'Бизнес-план для социального контракта', 'Подсобники разноробочие']

ЗАПРОС: сделать сайт под ключ | фильтры: Вид услуги Деловые услуги Тип услуги | recall: 1.0
   нужно: ['Создание сайта под ключ. Лендинг для бизнеса']
   топ-5: ['Создание сайта под ключ. Лендинг для бизнеса', 'Анимация для сайтов - Webflowприложениядля TildaUI', 'Поддержка и доработка сайтов по договору', 'Создание бота с ии, Telegram, Max', 'Создание и разработка сайтов']



Разбор промахов: для объявлений, которые не попали в топ-50, смотрю, насколько вероятна была их локация.

In [ ]:
missed = []
for _, row in val_b.iterrows():
    for item_id in row['relevant']:
        if item_id not in set(row['pred']):
            missed.append({'query': row['search_query'], 'search_location_id': row['search_location_id'], 'item_id': item_id})
missed = pd.DataFrame(missed)
missed = missed.merge(all_corpus[['item_id', 'item_title_raw', 'item_location_id']], on='item_id', how='left')
pair_counts = train_part.groupby(['search_location_id', 'item_location_id']).size()
search_counts = train_part.groupby('search_location_id').size()
pair_n = pair_counts.reindex(list(zip(missed['search_location_id'], missed['item_location_id']))).fillna(0).values
missed['p_loc'] = pair_n / search_counts.reindex(missed['search_location_id']).fillna(1).values
print('промахов:', len(missed), 'из', val_b['relevant'].str.len().sum())
print('доля промахов, где P(локация объявления | локация поиска) < 0.01:', round((missed['p_loc'] < 0.01).mean(), 3))
print('доля промахов, где такой пары локаций вообще не было в train:', round((missed['p_loc'] == 0).mean(), 3))
missed.sample(12, random_state=0)[['query', 'item_title_raw', 'p_loc']]

промахов: 310 из 3325
доля промахов, где P(локация объявления | локация поиска) < 0.01: 0.477
доля промахов, где такой пары локаций вообще не было в train: 0.242


,query,item_title_raw,p_loc
300,утепление стен,Ремонт домов. Замена венцов. Подъем дома,0.411765
145,изготовление таблички шильдики мотоцикл,"Наклейки, шильдики, таблички",0.010296
302,сенокос,Сенокошение,0.016317
222,романтическое свидание,Поздравление из Африки,0.984696
52,монтаж кму,Переоборудование Грузовых Машин,0.007407
279,автовоз иркутск москва,"Автовоз в/из Красноярск, Перегон авто по РФ",0.000093
22,торт на день рождения детский,Гелиевые шарики с доставкой,0.944871
171,чистка ковролина офис,"Химчистка диванов, ковров на дому. Частник",0.920977
191,фотограф английский,Фотограф для портфолио,0.920977
245,отдых на природе домик снять,Организация отдыха на берегу реки,0.000000


Выводы по валидации:

- Один BM25 дает около 0.42. Самый большой прирост дает локация (до 0.85-0.87): одного текста мало, потому что одинаковых услуг в разных городах тысячи. Если после подбора весов убрать локацию, Recall@50 падает с 0.91 до 0.71.
- Фильтры из `search_infm_params_text` добавляют около 1.5-2 пунктов, подкатегория по похожим запросам, эмбеддинги, расширение запроса и расстояние - еще от 0.3 до 1 пункта каждый (таблица "без ...").
- После подбора весов BM25 по заголовку получил вес 0: заголовок уже входит в `bm_full`, а смысл заголовка лучше ловят эмбеддинги.
- Категория 114 как отдельный признак ничего не дает (вес 0): объявления из других категорий и так проигрывают по тексту и подкатегории. Жесткого фильтра по категории нет, так что редкие категории не выкидываются совсем.
- Итог: 0.921 на A (на ней подбирались веса) и 0.907 на отложенной B. На текстах, которые встречались в train, качество выше (0.92), на новых ниже (0.90). Хуже всего запросы из одного слова (около 0.84): под них подходят сотни объявлений в одном городе.
- Оставшиеся ошибки примерно поровну: объявление из "необычной" локации (почти половина промахов, часто доставка, перевозки по всей России, онлайн-услуги) и объявление, которое по тексту похоже на запрос слабо (например, "утепление стен" -> "Ремонт домов. Замена венцов").

## 9. Финальный ответ для benchmark

Теперь индекс строится по всему train, корпус - только `benchmark_items`, веса - подобранные выше.

In [ ]:
t = time.time()
bench_corpus = all_corpus.iloc[:n_bench_items].reset_index(drop=True)
final_index = build_index(train, bench_corpus, item_emb[:n_bench_items])
predictions = run(final_index, queries, {'best': best_w})['best']
print('предсказания готовы за', round(time.time() - t), 'сек')

answer = pd.DataFrame({
    'query_id': queries['query_id'].values,
    'answer': [' '.join(top50) for top50 in predictions],
})
print(answer.dtypes)
answer.to_csv('answer.csv', index=False)
answer.head()

предсказания готовы за 174 сек
query_id    str
answer      str
dtype: object


,query_id,answer
0,70DfDUpwjxB4lzFd,14d535285758af89 9515d1e1ecdac72c 355392014208b7bf 6042fecdd7753112 255fbeaf526a1cc1 d722bcda1a5...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 1c6d2079e07f4497 cbeccbecb1fb8d86 73ab3b0d708ea15e 367af128a9ea2a48 396da9dc8f9...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 3b370cc603f67947 d8fce513e4f000a7 3f89b8062dc85f1c dab52187b4500d9b edecb3695ab...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 da18e5119a944f0b 67b2cd847669eb36 3e76e71f9e7867a5 4c15e7abba063ac5 1e5924dc546...
4,YgHcM9MVbxKnxD1e,1a62634394544781 615c73ea4c3bfd15 9c41fe789f98d702 53f9c8caeeb8111e b8603925d28a8c99 60e3b99111b...


## 10. Проверки ответа (чек-лист)

In [ ]:
check = pd.read_csv('answer.csv', dtype=str, keep_default_na=False, encoding='utf-8')
with open('answer.csv', encoding='utf-8') as f:
    header = f.readline().strip()
lists = check['answer'].str.split(' ')
corpus_ids = set(items['item_id'])
answer_ids = set(i for lst in lists for i in lst)

checks = {
    'заголовок ровно query_id,answer': header == 'query_id,answer',
    'ровно две колонки': check.columns.tolist() == ['query_id', 'answer'],
    'строк столько же, сколько запросов': len(check) == len(queries),
    'все query_id из benchmark есть': set(check['query_id']) == set(queries['query_id']),
    'нет лишних query_id': set(check['query_id']) <= set(queries['query_id']),
    'нет повторов query_id': check['query_id'].is_unique,
    'query_id по 16 символов': check['query_id'].str.len().eq(16).all(),
    'не больше 50 item_id в строке': lists.str.len().le(50).all(),
    'нет пустых ответов': lists.str.len().ge(1).all(),
    'нет повторов item_id в строке': all(len(x) == len(set(x)) for x in lists),
    'все item_id есть в корпусе': answer_ids <= corpus_ids,
    'item_id через один пробел, только 0-9a-f': check['answer'].str.fullmatch(r'[0-9a-f]{16}( [0-9a-f]{16})*').all(),
    'файл совпадает с тем, что было в памяти': check['query_id'].tolist() == answer['query_id'].tolist() and check['answer'].tolist() == answer['answer'].tolist(),
}
for name, ok in checks.items():
    print('OK  ' if ok else 'FAIL', name)
assert all(checks.values())
print('item_id на запрос:', lists.str.len().value_counts().to_dict())

OK   заголовок ровно query_id,answer
OK   ровно две колонки
OK   строк столько же, сколько запросов
OK   все query_id из benchmark есть
OK   нет лишних query_id
OK   нет повторов query_id
OK   query_id по 16 символов
OK   не больше 50 item_id в строке
OK   нет пустых ответов
OK   нет повторов item_id в строке
OK   все item_id есть в корпусе
OK   item_id через один пробел, только 0-9a-f
OK   файл совпадает с тем, что было в памяти
item_id на запрос: {50: 2452}


Пограничные случаи на запросах benchmark.

In [ ]:
bench = queries.copy()
bench['pred'] = pd.Series(predictions, index=bench.index)
bench['n_words'] = bench['search_query'].str.split().str.len()
bench_bm25 = (query_matrix(bench['query_st'], final_index['vocab_full']) @ final_index['bm_full']).tocsr()
bench['bm25_found'] = np.diff(bench_bm25.indptr)
bench['odd_symbols'] = bench['search_query'].str.contains(r'[^a-zа-яё0-9 ]', case=False)
empty_params = bench['search_infm_params_text'] == ''

print('пустой search_infm_params_text:', empty_params.sum(), 'запросов, кандидатов у них min =', bench.loc[empty_params, 'pred'].str.len().min())
print('запросы из одного слова:', (bench['n_words'] == 1).sum(), ', кандидатов min =', bench.loc[bench['n_words'] == 1, 'pred'].str.len().min())
print('запросы, где BM25 нашел меньше 50 объявлений:', (bench['bm25_found'] < 50).sum(), ', из них совсем 0:', (bench['bm25_found'] == 0).sum())
print('   у них кандидатов min =', bench.loc[bench['bm25_found'] < 50, 'pred'].str.len().min())
print('запросы с необычными символами:', bench['odd_symbols'].sum(), ', пример:', bench.loc[bench['odd_symbols'], 'search_query'].head(5).tolist())
print('локации поиска, которых нет в train:', (~bench['search_location_id'].isin(set(train['search_location_id']))).sum())

item_category = pd.Series(items['item_category_id'].values, index=items['item_id'].values)
answer_categories = item_category[[i for p in predictions for i in p]]
print('объявлений не из категории 114 в ответах:', (answer_categories != 114).sum(), 'из', len(answer_categories))

text_key = pd.Series((items['item_title_raw'] + '|' + items['item_description_raw']).values, index=items['item_id'].values)
dup_slots = sum(len(p) - text_key[p].nunique() for p in predictions)
print('мест в топ-50, занятых копиями одинакового текста:', dup_slots, 'из', sum(len(p) for p in predictions))

пустой search_infm_params_text: 1546 запросов, кандидатов у них min = 50
запросы из одного слова: 109 , кандидатов min = 50
запросы, где BM25 нашел меньше 50 объявлений: 33 , из них совсем 0: 1
   у них кандидатов min = 50
запросы с необычными символами: 31 , пример: ['маникюр ул. выборная', 'экспертиза наро-фоминский', 'репетиторство русский,математика 4 класс', 'установка сплит-систем', 'карта т-банк 1500']
локации поиска, которых нет в train: 0


объявлений не из категории 114 в ответах: 964 из 122600


мест в топ-50, занятых копиями одинакового текста: 933 из 122600


По пограничным случаям:

- пустые фильтры и NaN в текстах не ломают пайплайн: пропуски заменены на пустые строки, у таких запросов просто нулевой сигнал `flt`;
- скор считается для всех объявлений корпуса, поэтому даже если по словам ничего не нашлось, 50 кандидатов набираются по локации, эмбеддингам и подкатегории. Повторов нет, потому что `argpartition` возвращает разные позиции корпуса, а `item_id` в корпусе уникальны;
- жесткой фильтрации по категории нет, так что редкие категории не выкидываются, они просто получают меньший скор;
- копии одинаковых объявлений занимают около 1-2% мест. На валидации пробовал оставлять только одну копию текста в топ-50, Recall@50 не изменился, поэтому оставил как есть;
- токенизатор на эмодзи, цифрах и латинице не падает (проверено в разделе 3).

## 11. Воспроизводимость

Случайность есть только в сплите (зафиксирован `SEED = 42`). BM25, TF-IDF и косинусы детерминированы, модель эмбеддингов используется как есть, без дообучения. Ниже хэш файла, чтобы сравнить с отправленным `answer.csv` после повторного запуска.

In [ ]:
with open('answer.csv', 'rb') as f:
    print('md5 answer.csv:', hashlib.md5(f.read()).hexdigest())
print('весь ноутбук выполнился за', round((time.time() - start_time) / 60, 1), 'мин')

md5 answer.csv: 50fe37d207d69d364c109960ba74979d
весь ноутбук выполнился за 27.5 мин


## 12. Итог

**Данные и признаки.** Из запроса: текст, локация, фильтры `search_infm_params_text`. Из объявления: заголовок, описание, параметры, подкатегория, категория, локация и координаты. Train используется для статистики: переходы "локация поиска -> локация объявления", подкатегории и заголовки объявлений, выбранных по похожим запросам. Цену, рейтинг и флаги не использовал: они говорят о качестве объявления, а не о том, подходит ли оно под запрос, это скорее задача ранжирования.

**Алгоритм.** Для каждого запроса скор считается по всему корпусу как взвешенная сумма: BM25 (весь текст и заголовок), log P(локация объявления | локация поиска), штраф за расстояние, совпадение фильтров, log вероятности подкатегории по 20 похожим запросам из train, расширение запроса заголовками из train и косинус эмбеддингов `rubert-tiny-turbo`. Берутся 50 лучших.

**Как проверял.** Свой сплит train по уникальным комбинациям `search_*` полей, две выборки по ~3000 запросов с той же долей знакомых текстов, что и в benchmark (37%). Поиск по корпусу benchmark + релевантные объявления. Веса подбирались на A, итоговая оценка на B (значения в разделах 7-8).

**Ошибки и пограничные случаи.** Раздел 10: формат файла и типы ID, пустые фильтры, короткие запросы, запросы без совпадений по словам, редкие категории, дубли объявлений, необычные символы. Основные оставшиеся промахи разобраны в конце раздела 8.

**Что пробовал и не оставил.** TF-IDF по символьным n-граммам заголовка (на валидации только ухудшал), поиск похожих запросов по символьным n-граммам вместо эмбеддингов (прироста нет), вес локации в зависимости от "локальности" запроса (прироста нет), ограничение копий одинакового текста в топ-50 (прироста нет). Еще дообучал `rubert-tiny-turbo` на парах "запрос - заголовок" из обучающей части (одна эпоха на CPU, около 20 минут): сам по себе он стал заметно лучше исходной модели (Recall@50 вместе с локацией 0.83 против 0.71), но в общей смеси сигналов прироста не дал, при подборе весов его вес остался нулевым. Поэтому в финале модель используется без дообучения: так проще и быстрее воспроизводить.

**Open-source.** pandas, numpy, scipy, scikit-learn (CountVectorizer, TfidfVectorizer), snowballstemmer, sentence-transformers + PyTorch, модель `sergeyzh/rubert-tiny-turbo` с Hugging Face.